In [1]:
from datasets import load_dataset

dataset = load_dataset("sh0416/ag_news")

sample = dataset["train"][0]

sample_text = (
    sample["title"] + " " + sample["description"]
)

# Transformer text classification

Продолжаем классификацию AG News с помощью предобученного Transformer.

Используем DistilBERT и Hugging Face Transformers.

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

sample = dataset["train"][0]

sample_text = sample["title"] + " " + sample["description"]

encoded = tokenizer(
    sample_text,
    truncation=True,
    padding="max_length",
    max_length=100,
    return_tensors="pt",
    return_token_type_ids=False
)

print(encoded)

{'input_ids': tensor([[  101,  2813,  2358,  1012,  6468, 15020,  2067,  2046,  1996,  2304,
          1006, 26665,  1007, 26665,  1011,  2460,  1011, 19041,  1010,  2813,
          2395,  1005,  1055,  1040, 11101,  2989,  1032,  2316,  1997, 11087,
          1011, 22330,  8713,  2015,  1010,  2024,  3773,  2665,  2153,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
         0, 0

## 1. Токенизация DistilBERT

Используем готовый tokenizer модели `distilbert-base-uncased`.

Tokenizer преобразует текст в `input_ids` и создаёт `attention_mask`, которая показывает модели, какие позиции содержат реальные токены, а какие являются padding.

In [3]:
tokens = tokenizer.convert_ids_to_tokens(
    encoded["input_ids"][0]
)

print(tokens)

['[CLS]', 'wall', 'st', '.', 'bears', 'claw', 'back', 'into', 'the', 'black', '(', 'reuters', ')', 'reuters', '-', 'short', '-', 'sellers', ',', 'wall', 'street', "'", 's', 'd', '##wind', '##ling', '\\', 'band', 'of', 'ultra', '-', 'cy', '##nic', '##s', ',', 'are', 'seeing', 'green', 'again', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


## 2. Предобученная модель DistilBERT

Загрузим предобученный DistilBERT для классификации текста.

Модель уже обучена работать с английским языком. Для нашей задачи добавляется классификационный выход на 4 класса AG News.

In [4]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

outputs = model(
    input_ids=encoded["input_ids"],
    attention_mask=encoded["attention_mask"]
)

print(outputs.logits)
print(outputs.logits.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tensor([[-0.0681, -0.1711, -0.0281, -0.0845]], grad_fn=<AddmmBackward0>)
torch.Size([1, 4])


## 3. Подготовка данных для fine-tuning

Объединим заголовок и описание каждой новости в одно текстовое поле.

Эти тексты затем токенизируем с помощью tokenizer DistilBERT и используем для дообучения модели.

In [5]:
def combine_text(row):
    return {
        "text": row["title"] + " " + row["description"]
    }


dataset = dataset.map(combine_text)

print(dataset["train"][0]["text"])

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


## 4. Токенизация всего датасета

Преобразуем тексты всех новостей в формат DistilBERT.

Tokenizer создаст для каждой новости `input_ids` и `attention_mask`.

In [6]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=100,
        return_token_type_ids=False
    )


tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True
)

print(tokenized_dataset["train"][0].keys())

dict_keys(['label', 'title', 'description', 'text', 'input_ids', 'attention_mask'])


## 5. Train и validation

Разделим обучающую выборку на train и validation.

Train используется для обновления параметров модели, validation — для проверки качества во время обучения. Test оставляем отдельно для финальной оценки.

In [7]:
train_val_dataset = tokenized_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = train_val_dataset["train"]
val_dataset = train_val_dataset["test"]

test_dataset = tokenized_dataset["test"]

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

108000
12000
7600


## 6. Подготовка labels

Приведём номера классов из диапазона `1–4` к диапазону `0–3`, который используется моделью при многоклассовой классификации.

In [8]:
def normalize_label(row):
    return {
        "label": row["label"] - 1
    }


train_dataset = train_dataset.map(normalize_label)
val_dataset = val_dataset.map(normalize_label)
test_dataset = test_dataset.map(normalize_label)

print(train_dataset[0]["label"])

0


## 7. Формат данных для PyTorch

Оставим только те поля, которые нужны модели при обучении: `input_ids`, `attention_mask` и `label`, и будем получать их как PyTorch tensors.

In [9]:
columns = [
    "input_ids",
    "attention_mask",
    "label"
]

train_dataset.set_format(
    type="torch",
    columns=columns
)

val_dataset.set_format(
    type="torch",
    columns=columns
)

test_dataset.set_format(
    type="torch",
    columns=columns
)

print(train_dataset[0])

{'label': tensor(0), 'input_ids': tensor([  101, 13905,  1998,  4963,  1999,  2235,  2845,  2237,  2044,  6859,
         2022, 14540,  2319,  1010,  3607,  1006, 26665,  1007,  1011,  1996,
         4288,  1997,  2062,  2084, 13710,  2336,  1010,  3008,  1998,  5089,
         2076,  1996,  6703,  2203,  2000,  1037,  5187,  1011,  3178,  2082,
         6859,  2187,  4510,  1037,  2155, 22154,  1999,  1996,  2235,  2845,
         2237,  1997,  2022, 14540,  2319,  1012,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
      

## 8. Настройка fine-tuning

Зададим основные параметры дообучения DistilBERT: количество эпох, размер батча и learning rate.

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="distilbert-ag-news",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5
)

## 9. Trainer

Соберём модель, параметры обучения и подготовленные датасеты в `Trainer`.

`Trainer` будет выполнять обучение батчами, считать loss, делать `backward()` и обновлять параметры модели.

In [14]:
from transformers import Trainer
from sklearn.metrics import accuracy_score


def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predicted_classes = predictions.argmax(axis=1)

    return {
        "accuracy": accuracy_score(
            labels,
            predicted_classes
        )
    }

small_train_dataset = train_dataset.shuffle(seed=42).select(range(20000))
small_val_dataset = val_dataset.shuffle(seed=42).select(range(2000))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_val_dataset,
    compute_metrics=compute_metrics
)

## 10. Fine-tuning DistilBERT

Запустим дообучение DistilBERT на AG News.

Во время обучения `Trainer` будет брать батчи из train-датасета, считать loss, выполнять `backward()` и обновлять параметры модели.

In [15]:
trainer.train()

Step,Training Loss
500,0.287989
1000,0.257690


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.2672667724609375, metrics={'train_runtime': 1027.3397, 'train_samples_per_second': 19.468, 'train_steps_per_second': 1.217, 'total_flos': 517469232000000.0, 'train_loss': 0.2672667724609375, 'epoch': 1.0})

## 11. Оценка на validation

Проверим качество дообученной модели на validation-выборке.

Модель не будет изменять параметры — мы только посчитаем loss и accuracy на данных, которые не использовались для обучения.

In [16]:
val_results = trainer.evaluate(
    eval_dataset=small_val_dataset
)

print(val_results)

/Users/art/Desktop/internships/nlp-text-classification/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy
0.257690,0.255091,1250,0.917000


{'eval_loss': 0.25509098172187805, 'eval_accuracy': 0.917}


## 12. Финальная оценка на test

Проверим дообученную модель на официальной test-выборке AG News.

Эти данные не использовались ни для обучения, ни для выбора параметров модели.

In [17]:
test_results = trainer.evaluate(
    eval_dataset=test_dataset
)

print(test_results)

/Users/art/Desktop/internships/nlp-text-classification/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy
0.257690,0.239229,1250,0.920263


{'eval_loss': 0.2392289936542511, 'eval_accuracy': 0.9202631578947369}


## 13. Итоги

Сравнили три подхода к классификации AG News:

- TF-IDF + Logistic Regression — accuracy: 0.915
- PyTorch Embedding classifier — accuracy: 0.913
- Fine-tuned DistilBERT — accuracy: 0.920

DistilBERT показал лучший результат, несмотря на fine-tuning только на подвыборке из 20 000 train-примеров.

Классический TF-IDF baseline при этом остаётся очень сильным и значительно дешевле в обучении.